In [ ]:
print('Установка необходимых библиотек...')
!pip install -q --no-cache-dir 'gymnasium[box2d]' 'stable-baselines3' tensorboard moviepy > /dev/null 2>&1

In [ ]:
print('Импорт основных библиотек...')
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display, clear_output
import os
import numpy as np

In [ ]:
print('Создание папок для видео...')
os.makedirs("Videos/Accidents", exist_ok=True)
os.makedirs("Videos/Landings", exist_ok=True)
os.makedirs("Models", exist_ok=True)

In [ ]:
print('Демонстрация поведения случайного агента...')
random_environment = gym.make("LunarLander-v3", render_mode="rgb_array")
random_environment = RecordVideo(
    random_environment,
    video_folder="Videos/Accidents",
    episode_trigger=lambda episode_id: episode_id < 3,
    name_prefix="Random"
)

random_rewards = []

for episode in range(3):
    observation, info = random_environment.reset()
    episode_reward = 0
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action = random_environment.action_space.sample()
        observation, reward, terminated, truncated, info = random_environment.step(action)
        episode_reward += reward

    random_rewards.append(episode_reward)
    print(f'  Эпизод {episode + 1}: награда = {episode_reward:+.1f}')

random_environment.close()

print('\n')

print(f'Средняя награда случайного агента: {np.mean(random_rewards):+.1f}')
display(Video("Videos/Accidents/Random-episode-0.mp4", embed=True, width=600))

In [ ]:
print('Обучение агента с использованием алгоритма PPO...')
training_environment = make_vec_env("LunarLander-v3", n_envs=8)
trained_agent = PPO(
    policy="MlpPolicy",
    env=training_environment,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    tensorboard_log="./tensorboard_logs",
    seed=42
)

trained_agent.learn(total_timesteps=450_000, progress_bar=True)

trained_agent.save("Models/Lunar lander")

In [ ]:
print('Оценка качества обученного агента...')
evaluation_environment = gym.make("LunarLander-v3")
mean_reward, standart_reward = evaluate_policy(trained_agent, evaluation_environment, n_eval_episodes=10)

print(f'Средняя награда после обучения: {mean_reward:+.1f} ± {standart_reward:.1f}')
print('> 200 — успешная мягкая посадка')
print('> 250 — отличный результат')

print('\n')

print('Запись видео с идеальными посадками...')
testing_environment = gym.make("LunarLander-v3", render_mode="rgb_array")
testing_environment = RecordVideo(
    testing_environment,
    video_folder="Videos/Landings",
    episode_trigger=lambda episode_id: episode_id < 3,
    name_prefix="Result"
)

for episode in range(3):
    observation, info = testing_environment.reset()
    terminated = False
    truncated = False
    episode_reward = 0

    while not (terminated or truncated):
        action, _ = trained_agent.predict(observation, deterministic=True)
        observation, reward, terminated, truncated, info = testing_environment.step(action)
        episode_reward += reward

    print(f'  Эпизод {episode + 1}: награда = {episode_reward:+.1f}')

testing_environment.close()

display(Video("Videos/Landings/Result-episode-0.mp4", embed=True, width=600))